[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/ct_cross_backend_demo.ipynb)

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# Cross-Backend Analysis Composition with Interpretune

This notebook demonstrates **multi-model, cross-backend analysis composition** using Interpretune's
AnalysisStore and analysis ops framework. We combine:

- **Stage A — GPT-2 SAE Analysis (TransformerBridge):** SAE-based logit diff analysis using the
  `AnalysisRunner` with TransformerBridge mode
- **Stage B — Gemma-2 Circuit Analysis (CT NNsight):** Full circuit-tracer attribution and
  intervention pipeline using namespace op access (`it.op_name`)
- **Stage C — Cross-Analysis Enrichment (CPU only):** Combining results from both backends
  into a unified analysis summary

**Key concepts demonstrated:**
- TransformerBridge mode (TL v3, no weight duplication)
- AnalysisRunner for automated analysis workflows
- Namespace op access for fine-grained control (`it.op_name`)
- AnalysisStore persistence and reload
- Multi-model, cross-backend result composition
- Feature intervention with logit comparison

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
backend = "nnsight"  # CT backend: "transformerlens" or "nnsight"
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
intervention_scale_factor = 10.0  # Multiplier for feature intervention amplitudes

In [ ]:
# @title Imports { display-mode: "form" }
import gc
import tempfile
import torch
from pathlib import Path

import interpretune as it
from it_examples import _ACTIVE_PATCHES  # noqa: F401
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY
from it_examples.utils.example_helpers import required_os_env
from interpretune import ITSessionConfig, ITSession, AnalysisRunner, AnalysisCfg, AnalysisStore
from interpretune.analysis.core import LatentAnalysisTargets
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.config import init_analysis_cfgs
from interpretune.config.sae_lens import SAELensFromPretrainedConfig

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to "/full/path/to/.env" to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

In [ ]:
# Create a shared temp directory for cross-backend Store persistence
cross_backend_dir = Path(tempfile.mkdtemp(prefix="ct_cross_backend_"))
store_a_dir = cross_backend_dir / "store_a_gpt2_sae"
print(f"Cross-backend analysis directory: {cross_backend_dir}")

---
## Stage A — GPT-2 SAE Analysis (TransformerBridge)

In this stage we load GPT-2 using TransformerBridge mode (TL v3) with the SAE-Lens adapter.
TransformerBridge wraps the HuggingFace model directly without weight conversion, providing
better memory efficiency and HF ecosystem compatibility.

We use the `AnalysisRunner` to automatically orchestrate a `logit_diffs_base` analysis — computing
logit differences between correct and incorrect answer tokens across dataset batches. The results
are saved to an `AnalysisStore` on disk for later cross-backend composition.

In [ ]:
# Load the GPT-2 SAE demo configuration (uses TransformerBridge by default)
base_itdm_cfg_a, base_it_cfg_a, dm_cls_a, m_cls_a = MODULE_EXAMPLE_REGISTRY.get("gpt2.rte_demo.sae_lens")

# Optionally override core_log_dir
if core_log_dir:
    base_it_cfg_a.core_log_dir = core_log_dir

# Configure SAE analysis targets (attention SAEs on layers 9-10)
sae_targets = LatentAnalysisTargets(sae_release="gpt2-small-hook-z-kk", target_layers=[9, 10])
sae_cfgs = [
    SAELensFromPretrainedConfig(release=sae_fqn.release, sae_id=sae_fqn.sae_id)
    for sae_fqn in sae_targets.latent_model_fqns
]
base_it_cfg_a.sae_cfgs = sae_cfgs

# Create session with core + sae_lens adapters (TransformerBridge mode)
session_cfg_a = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.sae_lens),
    datamodule_cfg=base_itdm_cfg_a,
    module_cfg=base_it_cfg_a,
    datamodule_cls=dm_cls_a,
    module_cls=m_cls_a,
)

it_session_a = ITSession(session_cfg_a)
print("Stage A: GPT-2 SAE session created")

In [ ]:
# Initialize the session (loads GPT-2 model with TransformerBridge)
it.it_init(**it_session_a)
module_a = it_session_a.module
print(f"Stage A model type: {type(module_a.model).__name__}")
print(f"Stage A model device: {next(module_a.model.parameters()).device}")

In [ ]:
# Set up the AnalysisRunner for logit_diffs_base
run_kwargs_a = dict(latent_analysis_targets=sae_targets, it_session=it_session_a, max_epochs=1)
run_config_a = dict(
    limit_analysis_batches=3,
    ignore_manual=True,
    op_output_dataset_path=str(store_a_dir),
    **run_kwargs_a,
)
runner_a = AnalysisRunner(run_cfg=run_config_a)

# Define the analysis configuration: logit_diffs_base (clean forward, no SAE)
logit_diffs_base_cfg = AnalysisCfg(target_op=it.logit_diffs_base, save_prompts=True, save_tokens=False)

# Run the analysis — this orchestrates the full batch loop and saves results to disk
print("Running GPT-2 logit_diffs_base analysis...")
store_a = runner_a.run_analysis(analysis_cfgs=(logit_diffs_base_cfg,))
print(f"Stage A analysis complete! Store saved to: {store_a_dir}")
print(f"Store A columns: {store_a.dataset.column_names}")
print(f"Store A rows: {len(store_a.dataset)}")

In [ ]:
# Display a sample of the logit diffs results
tokenizer_a = module_a.datamodule.tokenizer
for i in range(min(3, len(store_a.dataset))):
    row = store_a.dataset[i]
    prompt = row.get("prompts", "N/A")
    logit_diff = row.get("logit_diffs", None)
    if logit_diff is not None:
        import torch

        if isinstance(logit_diff, torch.Tensor):
            if logit_diff.dim() == 0:
                logit_diff = logit_diff.item()
            else:
                logit_diff = logit_diff.tolist()
        print(f"  Sample {i}: logit_diff={logit_diff}  prompt={prompt!r:.60}")
    else:
        print(f"  Sample {i}: {list(row.keys())}")

In [ ]:
# Free GPU memory for Stage B
del runner_a, module_a, it_session_a, session_cfg_a
del base_itdm_cfg_a, base_it_cfg_a, dm_cls_a, m_cls_a
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after Stage A teardown: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
else:
    print("Stage A teardown complete (CPU mode)")

---
## Stage B — Gemma-2 Circuit Analysis (CT NNsight)

In this stage we load Gemma-2-2b with the circuit-tracer adapter using the NNsight execution backend.
We use the `attribution_from_concept` composite op to run the first four analysis steps as a pipeline,
then call `feature_intervention_forward` separately with actual vocabulary token IDs:

| Step | Op | Purpose |
|------|----|---------|
| Pipeline | `concept_direction` | Compute a directional concept vector via paired rejection |
| Pipeline | `compute_attribution_graph` | Build a transcoder-based attribution graph |
| Pipeline | `graph_node_influence` | Score graph nodes by causal influence on the concept |
| Pipeline | `extract_top_features` | Extract the most influential features from the graph |
| Separate | `feature_intervention_forward` | Amplify top features and measure prediction shifts |

In [ ]:
# Load the Gemma-2 circuit-tracer demo configuration
base_itdm_cfg_b, base_it_cfg_b, dm_cls_b, m_cls_b = MODULE_EXAMPLE_REGISTRY.get("gemma2.rte_demo.circuit_tracer")

# Configure backend
base_it_cfg_b.circuit_tracer_cfg.backend = backend
if backend == "nnsight":
    from interpretune.config.nnsight import NNsightConfig

    base_it_cfg_b.nnsight_cfg = NNsightConfig(
        model_name="google/gemma-2-2b",
        device_map="cuda",
        torch_dtype="bfloat16",
        dispatch=True,
    )

# Optionally override core_log_dir
if core_log_dir:
    base_it_cfg_b.core_log_dir = core_log_dir

# Use Gemma Scope transcoders (matches upstream attribution_targets_demo)
base_it_cfg_b.circuit_tracer_cfg.transcoder_set = "gemma"

# Configure intervention settings
base_it_cfg_b.circuit_tracer_cfg.intervention_value_source = "top_feature_activation_values"
base_it_cfg_b.circuit_tracer_cfg.intervention_scale_factor = intervention_scale_factor

# Select adapter composition based on backend
if backend == "nnsight":
    adapter_ctx_b = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    adapter_ctx_b = (it.Adapter.core, it.Adapter.circuit_tracer)

session_cfg_b = ITSessionConfig(
    adapter_ctx=adapter_ctx_b,
    datamodule_cfg=base_itdm_cfg_b,
    module_cfg=base_it_cfg_b,
    datamodule_cls=dm_cls_b,
    module_cls=m_cls_b,
)

it_session_b = ITSession(session_cfg_b)
print("Stage B: Gemma-2 CT session created")

In [ ]:
# Initialize the session (loads Gemma-2-2b model)
it.it_init(**it_session_b)
module_b = it_session_b.module
tokenizer_b = module_b.replacement_model.tokenizer

# Set up analysis config on module (required for ops that reference module.analysis_cfg)
graph_op = it.compute_attribution_graph
module_b.analysis_cfg = AnalysisCfg(target_op=graph_op, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module_b, [module_b.analysis_cfg])

print(f"Stage B model: {type(module_b.replacement_model).__name__}")

In [ ]:
# Run concept direction + attribution graph + node influence + feature extraction as a pipeline
pipeline = it.attribution_from_concept

# Define concept groups and prompt
capitals = ["▁Austin", "▁Sacramento", "▁Olympia", "▁Atlanta"]
states = ["▁Texas", "▁California", "▁Washington", "▁Georgia"]
concept_label = "Concept: Capitals − States"

prompt_b = "Fact: the capital of the state containing Dallas is"
n_top = 10

pipeline_result = pipeline(
    module_b,
    AnalysisBatch(
        concept_group_a=capitals,
        concept_group_b=states,
        concept_label=concept_label,
        concept_direction_mode="paired_rejection",
        prompts=[prompt_b],
    ),
    None,
    0,
    top_n=n_top,
)

print(f"Concept direction shape: {pipeline_result.concept_direction.shape}")
print(f"\nTop {n_top} features by influence:")
print(f"{'Rank':<6} {'Layer':<8} {'Position':<10} {'Feature':<10} {'Activation':<12} {'Score':<10}")
print("-" * 56)
for idx in range(n_top):
    feat = pipeline_result.top_feature_ids[idx]
    print(
        f"{idx + 1:<6} {feat[0].item():<8} {feat[1].item():<10} {feat[2].item():<10} "
        f"{pipeline_result.top_feature_activation_values[idx].item():<12.4f} "
        f"{pipeline_result.top_feature_scores[idx].item():<10.4f}"
    )

In [ ]:
# Run feature intervention — amplify top features and measure logit changes
austin_id = tokenizer_b.encode("▁Austin", add_special_tokens=False)[-1]
dallas_id = tokenizer_b.encode("▁Dallas", add_special_tokens=False)[-1]

intervention_op = it.feature_intervention_forward
intervention_result = intervention_op(
    module_b,
    AnalysisBatch(
        prompts=[prompt_b],
        top_feature_ids=pipeline_result.top_feature_ids,
        top_feature_activation_values=pipeline_result.top_feature_activation_values,
        top_feature_scores=pipeline_result.top_feature_scores,
        logit_target_ids=torch.tensor([austin_id, dallas_id], dtype=torch.long),
    ),
    None,
    0,
    intervention_scale_factor=intervention_scale_factor,
)

pre_logits = intervention_result.pre_intervention_logits.float().cpu()
post_logits = intervention_result.post_intervention_logits.float().cpu()

pre_gap = (pre_logits[austin_id] - pre_logits[dallas_id]).item()
post_gap = (post_logits[austin_id] - post_logits[dallas_id]).item()

print(
    f"Pre-intervention:  Austin={pre_logits[austin_id].item():.4f}, "
    f"Dallas={pre_logits[dallas_id].item():.4f}, gap={pre_gap:+.4f}"
)
print(
    f"Post-intervention: Austin={post_logits[austin_id].item():.4f}, "
    f"Dallas={post_logits[dallas_id].item():.4f}, gap={post_gap:+.4f}"
)
print(f"Gap change: {post_gap - pre_gap:+.4f}")

In [ ]:
# Collect Stage B results into a summary dict for cross-analysis
stage_b_summary = {
    "model": "google/gemma-2-2b",
    "backend": backend,
    "prompt": prompt_b,
    "concept_label": concept_label,
    "n_top_features": n_top,
    "top_features": [
        {
            "rank": i + 1,
            "layer": pipeline_result.top_feature_ids[i][0].item(),
            "position": pipeline_result.top_feature_ids[i][1].item(),
            "feature_idx": pipeline_result.top_feature_ids[i][2].item(),
            "activation": pipeline_result.top_feature_activation_values[i].item(),
            "score": pipeline_result.top_feature_scores[i].item(),
        }
        for i in range(len(pipeline_result.top_feature_ids))
    ],
    "pre_intervention_logits": {
        "Austin": pre_logits[austin_id].item(),
        "Dallas": pre_logits[dallas_id].item(),
    },
    "post_intervention_logits": {
        "Austin": post_logits[austin_id].item(),
        "Dallas": post_logits[dallas_id].item(),
    },
    "logit_delta_austin": (post_logits[austin_id] - pre_logits[austin_id]).item(),
    "logit_delta_dallas": (post_logits[dallas_id] - pre_logits[dallas_id]).item(),
    "scale_factor": intervention_scale_factor,
}
print(f"Stage B summary collected: {len(stage_b_summary['top_features'])} features, intervention deltas recorded")

In [ ]:
# Free GPU memory
del module_b, it_session_b, session_cfg_b
del base_itdm_cfg_b, base_it_cfg_b, dm_cls_b, m_cls_b
del pipeline_result, intervention_result
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after Stage B teardown: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
else:
    print("Stage B teardown complete (CPU mode)")

---
## Stage C — Cross-Analysis Enrichment (CPU only)

In this final stage we combine results from both analyses — **no model is loaded**. We:

1. Reload the GPT-2 SAE `AnalysisStore` from disk (demonstrating Store persistence)
2. Combine with the in-memory Gemma-2 circuit analysis summary
3. Present a unified cross-analysis view comparing both frameworks' findings

This demonstrates that `AnalysisStore` data is fully backend-agnostic: results produced by different
models, adapters, and execution backends can all be loaded and compared on CPU.

In [ ]:
# Reload Stage A's AnalysisStore from disk (no model needed)
from datasets import load_from_disk

store_a_reloaded = AnalysisStore(dataset=load_from_disk(str(store_a_dir)))
print(f"Reloaded Store A from: {store_a_dir}")
print(f"  Columns: {store_a_reloaded.dataset.column_names}")
print(f"  Rows: {len(store_a_reloaded.dataset)}")

In [ ]:
# === Cross-Analysis Summary ===
print("=" * 72)
print("  CROSS-BACKEND ANALYSIS SUMMARY")
print("=" * 72)

# --- GPT-2 SAE Analysis (Store A) ---
print("\n--- Stage A: GPT-2 SAE Logit Diffs (TransformerBridge) ---")
print("  Model: openai-community/gpt2")
print("  Backend: TransformerBridge (TL v3)")
print("  Op: logit_diffs_base via AnalysisRunner")
print(f"  Batches analyzed: {len(store_a_reloaded.dataset)}")

# Extract logit diff statistics from Store A
logit_diffs_col = "logit_diffs"
if logit_diffs_col in store_a_reloaded.dataset.column_names:
    diffs = [store_a_reloaded.dataset[i][logit_diffs_col] for i in range(len(store_a_reloaded.dataset))]
    numeric_diffs = []
    for d in diffs:
        if d is None:
            continue
        import torch

        if isinstance(d, torch.Tensor):
            if d.dim() == 0:
                numeric_diffs.append(d.item())
            else:
                numeric_diffs.extend(d.tolist())
        else:
            numeric_diffs.append(float(d))
    if numeric_diffs:
        import statistics

        print(f"  Mean logit diff: {statistics.mean(numeric_diffs):+.4f}")
        print(f"  Std logit diff:  {statistics.stdev(numeric_diffs) if len(numeric_diffs) > 1 else 0:.4f}")
        print(f"  Range: [{min(numeric_diffs):+.4f}, {max(numeric_diffs):+.4f}]")
else:
    print(f"  Available columns: {store_a_reloaded.dataset.column_names}")

# --- Gemma-2 CT Analysis (Stage B in-memory) ---
print(f"\n--- Stage B: Gemma-2 Circuit Analysis (CT {backend}) ---")
print(f"  Model: {stage_b_summary['model']}")
print(f"  Backend: circuit-tracer ({stage_b_summary['backend']})")
print(f"  Concept: {stage_b_summary['concept_label']}")
print(f"  Prompt: '{stage_b_summary['prompt']}'")

# Feature layer distribution
layers = [f["layer"] for f in stage_b_summary["top_features"]]
unique_layers = sorted(set(layers))
print(f"\n  Top-{stage_b_summary['n_top_features']} Feature Distribution:")
for layer in unique_layers:
    count = layers.count(layer)
    print(f"    Layer {layer}: {count} feature(s)")

# Intervention effect
print(f"\n  Intervention (scale={stage_b_summary['scale_factor']}):")
print(
    f"    Austin logit: {stage_b_summary['pre_intervention_logits']['Austin']:.4f} → "
    f"{stage_b_summary['post_intervention_logits']['Austin']:.4f} "
    f"(Δ={stage_b_summary['logit_delta_austin']:+.4f})"
)
print(
    f"    Dallas logit: {stage_b_summary['pre_intervention_logits']['Dallas']:.4f} → "
    f"{stage_b_summary['post_intervention_logits']['Dallas']:.4f} "
    f"(Δ={stage_b_summary['logit_delta_dallas']:+.4f})"
)

# --- Cross-Framework Comparison ---
print("\n--- Cross-Framework Insights ---")
print("  Both analyses target the RTE task on different models/frameworks:")
print("  • GPT-2 + SAE-Lens: Measures per-sample logit differences between")
print("    correct and incorrect answer tokens across the dataset")
print("  • Gemma-2 + Circuit-Tracer: Identifies specific transcoder features")
print("    causally responsible for a concept and measures intervention effects")
print("\n  AnalysisStore enables this cross-framework composition because:")
print("  • Store A was produced by TL Bridge + SAE-Lens → reloaded on CPU ✓")
print("  • Stage B results from CT NNsight → accessible as plain Python dicts ✓")
print("  • No backend-specific code needed to read or combine results")
print("=" * 72)

---
## Summary

This notebook demonstrated **cross-backend analysis composition** with Interpretune:

| Stage | Model | Backend | Ops | Result |
|-------|-------|---------|-----|--------|
| A | GPT-2 | TransformerBridge + SAE-Lens | `logit_diffs_base` (via AnalysisRunner) | AnalysisStore on disk |
| B | Gemma-2-2b | Circuit-Tracer (NNsight) | `concept_direction` → `attribution_graph` → `intervention` | In-memory summary |
| C | — (CPU) | — | Cross-analysis enrichment | Unified comparison |

**Key takeaways:**
- **TransformerBridge** (TL v3) wraps HF models without weight duplication — used for GPT-2 SAE analysis
- **AnalysisRunner** automates batch-level analysis orchestration with persistent AnalysisStore output
- **Namespace op access** (`it.op_name`) provides fine-grained control over individual analysis ops
- **AnalysisStore** data is fully backend-agnostic: save from one backend, load and use anywhere
- **Sequential model loading** with `gc.collect()` keeps peak memory manageable for T4 Colab

### Next Steps

- See the [CT Analysis Backend Demo](ct_analysis_backend_demo.ipynb) for a deeper dive into the
  full circuit-tracer op pipeline
- See the [SAE-Lens Adapter Example](../saelens_adapter_example/saelens_adapter_example.ipynb) for
  more SAE analysis patterns
- Explore `it.DISPATCHER.registered_ops` to see all available analysis operations